### Imports and Paths

In [2]:
import re
from pathlib import Path
import pymupdf
import fitz

RAW_DIR = Path("data-raw")
OUT_DIR = Path("data/extracted")
OUT_DIR.mkdir(parents=True, exist_ok=True)

pdfs = sorted(RAW_DIR.glob("*.pdf"))
print(f"Found {len(pdfs)} PDFs:")
for p in pdfs:
    print(" -", p.name)

Found 6 PDFs:
 - sample_1.pdf
 - sample_2.pdf
 - sample_3.pdf
 - sample_4.pdf
 - sample_5.pdf
 - sample_6.pdf


### Extract text from every PDF

In [4]:
raw_texts = {}   # file stem -> full text
raw_pages = {}   # file stem -> list of page texts

for pdf in pdfs:
    doc = fitz.open(pdf)
    pages = [page.get_text("text") for page in doc]
    raw_pages[pdf.stem] = pages
    raw_texts[pdf.stem] = "\n".join(pages)
    (OUT_DIR / f"{pdf.stem}.txt").write_text(raw_texts[pdf.stem], encoding="utf-8")

print("Done. Extracted", len(raw_texts), "files")

Done. Extracted 6 files


### Health check for each file

In [6]:
for name, text in raw_texts.items():
    code = re.search(r"Document Code:\s*(\S+)", text)
    title = text.strip().split("\n")[0] if text.strip() else "(empty)"

    # Ignore table-of-contents entries, which look like "[Section N:"
    found = {int(n) for n in re.findall(r"(?<!\[)Section\s+(\d+):", text)}
    missing = [n for n in range(1, 24) if n not in found]

    empty_pages = sum(1 for p in raw_pages[name] if len(p.strip()) < 20)

    print("=" * 70)
    print("File:        ", name)
    print("Title:       ", title)
    print("Doc code:    ", code.group(1) if code else "NOT FOUND")
    print("Pages:       ", len(raw_pages[name]), f"(empty/scanned: {empty_pages})")
    print("Characters:  ", len(text))
    print("'■' symbols: ", text.count("■"))
    print("FAQ items:   ", len(re.findall(r"^Q\d{3}:", text, flags=re.M)))
    print("Sections 1-23 missing:", missing if missing else "none")

File:         sample_1
Title:        FinBase Fixed Deposits & Wealth Products Master Guidelines
Doc code:     FB-POL-FDW-2026-V3
Pages:        25 (empty/scanned: 0)
Characters:   81745
'■' symbols:  0
FAQ items:    100
Sections 1-23 missing: [21]
File:         sample_2
Title:        FinBase Payments, UPI & Refund Settlement SOP
Doc code:     FB-SOP-PAY-2026-V4
Pages:        25 (empty/scanned: 0)
Characters:   80684
'■' symbols:  0
FAQ items:    100
Sections 1-23 missing: [22]
File:         sample_3
Title:        FinBase Credit Cards Comprehensive Handbook & Cardholder
Doc code:     FB-POL-CC-2026-V5
Pages:        26 (empty/scanned: 0)
Characters:   93987
'■' symbols:  0
FAQ items:    100
Sections 1-23 missing: [22]
File:         sample_4
Title:        FinBase Digital Savings Accounts & Banking Operations Manual
Doc code:     FB-POL-SAV-2026-V3
Pages:        25 (empty/scanned: 0)
Characters:   81272
'■' symbols:  0
FAQ items:    100
Sections 1-23 missing: [22]
File:         sample_5
Tit

In [7]:
name = list(raw_texts.keys())[0]
print(name)
print("-" * 70)
print("\n".join(raw_texts[name].split("\n")[:40]))

sample_1
----------------------------------------------------------------------
FinBase Fixed Deposits & Wealth Products Master Guidelines
Document Code: FB-POL-FDW-2026-V3
Effective Date: October 1, 2026
Issuing Authority: FinBase Wealth Management & Treasury Committee
Classification: Official Customer Product Disclosure & Terms
Table of Contents
• [Section 1: Wealth Management Clause 1](#section-1)
• [Section 2: Wealth Management Clause 2](#section-2)
• [Section 3: Wealth Management Clause 3](#section-3)
• [Section 4: Wealth Management Clause 4](#section-4)
• [Section 5: Wealth Management Clause 5](#section-5)
• [Section 6: Wealth Management Clause 6](#section-6)
• [Section 7: Wealth Management Clause 7](#section-7)
• [Section 8: Wealth Management Clause 8](#section-8)
• [Section 9: Wealth Management Clause 9](#section-9)
• [Section 10: Wealth Management Clause 10](#section-10)
• [Section 11: Wealth Management Clause 11](#section-11)
• [Section 12: Wealth Management Clause 12](#secti

In [8]:
text = raw_texts[name]
i = text.find("Tenure Bucket")
if i == -1:
    i = text.find("Protocol ID")
if i == -1:
    i = text.find("Fee Item")
print(text[max(i - 200, 0): i + 800])

ection-23)
Section 1: Fixed Deposit Tenures, Interest Rates, and Senior Citizen Advantage
FinBase offers digital Fixed Deposits starting with a minimum deposit of I1,000 up to I50,00,000 per booking:
Tenure Bucket
Regular Customer Rate (p.a.)
Senior Citizen Rate (60+ yrs)
Compounding Frequency
7 days to 180 days
5.50% p.a.
6.00% p.a.
At Maturity
181 days to 364 days
6.25% p.a.
6.75% p.a.
Quarterly Compounded
1 year to less than 2 years
7.50% p.a.
8.00% p.a.
Quarterly Compounded
2 years to 3 years
7.25% p.a.
7.75% p.a.
Quarterly Compounded
3 years to 5 years
7.00% p.a.
7.50% p.a.
Quarterly Compounded

• Senior Citizen Rate Advantage: Senior citizen depositors aged 60 years and above are entitled to an additional interest rate of **0.50%
p.a.** across all deposit tenures.
Section 2: Premature Liquidation and Penalties
• Premature Withdrawal Permitted: Customers may break or liquidate their Fixed Deposit at any time through the FinBase App.
• Premature Penalty: A penalty of **1.00%** is d

In [9]:
import fitz
print("find_tables available:", hasattr(fitz.Page, "find_tables"))

find_tables available: True


### extraction helpers

In [11]:
import re, json
from pathlib import Path
import fitz

RAW_DIR = Path("data-raw")
PROC_DIR = Path("data/processed")
PROC_DIR.mkdir(parents=True, exist_ok=True)


def render_table(rows):
    lines = []
    for row in rows:
        cells = [re.sub(r"\s+", " ", c or "").strip() for c in row]
        if any(cells):
            lines.append(" | ".join(cells))
    return "\n".join(lines)


def page_to_text(page):
    found = page.find_tables().tables
    boxes = [fitz.Rect(t.bbox) for t in found]
    items = [(box.y0, box.x0, render_table(t.extract())) for t, box in zip(found, boxes)]
    for x0, y0, x1, y1, txt, _, btype in page.get_text("blocks"):
        if btype != 0:
            continue
        centre = fitz.Point((x0 + x1) / 2, (y0 + y1) / 2)
        if any(b.contains(centre) for b in boxes):
            continue  # this text is inside a table, already rendered above
        items.append((y0, x0, txt.strip()))
    items.sort(key=lambda it: (round(it[0]), it[1]))
    return "\n".join(t for _, _, t in items if t), len(found) 

### extract all PDFs

In [13]:
pdfs = sorted(p for p in RAW_DIR.iterdir() if p.suffix.lower() == ".pdf")
assert pdfs, f"No PDFs found in {RAW_DIR.resolve()}. Fix the working folder first."

raw = {}
for pdf in pdfs:
    doc = fitz.open(pdf)
    parts, n_tables = [], 0
    for i, page in enumerate(doc, start=1):
        text, n = page_to_text(page)
        n_tables += n
        parts.append(f"[[PAGE {i}]]\n{text}")
    raw[pdf.stem] = "\n".join(parts)
    print(f"{pdf.stem}: {len(doc)} pages, {n_tables} tables detected")

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
sample_1: 25 pages, 17 tables detected
sample_2: 25 pages, 20 tables detected
sample_3: 26 pages, 20 tables detected
sample_4: 25 pages, 22 tables detected
sample_5: 26 pages, 16 tables detected
sample_6: 25 pages, 15 tables detected


### cleaning function

In [15]:
def clean_text(text):
    text = text.replace("**", "").replace("■", "₹")
    text, n_rupee = re.subn(r"(?<![A-Za-z0-9])I(?=\d)", "₹", text)   # "I1,000" -> "₹1,000"
    text = re.sub(r"[ \t]+", " ", text)
    # a "Section N:" heading glued to the end of the previous sentence -> its own line
    text = re.sub(r"(?<=\S) (?=Section [A-Za-z0-9\-]+: )", "\n", text)
    text = re.sub(r"^•\s*\[Section[^\n]*$", "", text, flags=re.M)      # table of contents entries
    text = re.sub(r"^Table of Contents\s*$", "", text, flags=re.M)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip(), n_rupee

### section splitter and FAQ parser

In [17]:
HEADING = re.compile(r"^Section\s+([A-Za-z0-9\-]+):\s*(.*)$")
PAGE = re.compile(r"^\[\[PAGE (\d+)\]\]$")
Q_START = re.compile(r"^(Q\d{3}):\s*(.*)$")


def split_sections(text):
    sections = []
    cur = {"section_id": "Header", "title": "Document header", "page": 1, "lines": []}
    page = 1
    for line in text.split("\n"):
        line = line.strip()
        m = PAGE.match(line)
        if m:
            page = int(m.group(1))
            continue
        h = HEADING.match(line)
        if h:
            sections.append(cur)
            cur = {"section_id": h.group(1), "title": h.group(2).strip(), "page": page, "lines": []}
            continue
        if line:
            cur["lines"].append(line)
    sections.append(cur)
    for s in sections:
        s["text"] = "\n".join(s.pop("lines")).strip()
    return [s for s in sections if s["text"] or s["section_id"] != "Header"]


def title_key(title):
    return re.sub(r"\d+", "#", title).strip().lower()


def parse_faq(section):
    items, cur = [], None
    for line in section["text"].split("\n"):
        m = Q_START.match(line)
        if m:
            if cur:
                items.append(cur)
            cur = {"faq_id": m.group(1), "q": m.group(2).strip(), "lines": [], "page": section["page"]}
        elif cur is not None:
            cur["lines"].append(line)
    if cur:
        items.append(cur)

    parsed = []
    for it in items:
        q, lines = it["q"], list(it["lines"])
        tries = 0
        while not re.search(r"[?)]\s*$", q) and lines and tries < 2:   # question wrapped onto next line
            q += " " + lines.pop(0)
            tries += 1
        q = re.sub(r"\s*\([^()]*\d+\)\s*$", "", q).strip()             # drop "(Wealth product query 7)"
        answer, bullets = [], []
        for l in lines:
            if l.startswith("•"):
                bullets.append(l)
            elif bullets:
                bullets[-1] += " " + l
            else:
                answer.append(l)
        parsed.append({"faq_id": it["faq_id"], "question": q, "answer": " ".join(answer),
                       "details": bullets, "page": it["page"]})
    return parsed


def dedupe_faqs(faqs):
    seen, unique = {}, []
    for f in faqs:
        ans_key = re.sub(r"\[Reference[^\]]*\]", "", f["answer"]).lower()
        key = (f["question"].lower(), re.sub(r"\s+", " ", ans_key).strip())
        if key in seen:
            seen[key]["duplicate_ids"].append(f["faq_id"])
        else:
            f["duplicate_ids"] = []
            seen[key] = f
            unique.append(f)
    return unique

### parse every document and save

In [19]:
VALUE = re.compile(
    r"₹\s?[\d,\.]+|\d+(?:\.\d+)?\s?%|"
    r"\d+\s?(?:calendar |business |working )?(?:days?|years?|hours?|months?|mins?|minutes?|swipes?|attempts?)",
    re.I,
)


def signature(sec):
    """Digit-free sorted word list + the real values (amounts, %, durations)."""
    body = re.sub(r"\d+", "#", sec["text"]).lower()
    words = " ".join(sorted(re.sub(r"[\s|]+", " ", body).split()))
    values = tuple(sorted(v.lower().replace(" ", "") for v in VALUE.findall(sec["text"])))
    return words, values


def parse_doc(stem, raw_text):
    text, n_rupee = clean_text(raw_text)
    lines = [l.strip() for l in text.split("\n")]

    head = []
    for l in lines:
        if PAGE.match(l):
            continue
        if l.startswith("Document Code:") or HEADING.match(l):
            break
        head.append(l)
    doc_title = " ".join(x for x in head if x)
    code = re.search(r"Document Code:\s*(\S+)", text)
    doc_code = code.group(1) if code else stem

    sections = split_sections(text)
    faq_secs = [s for s in sections if "faq" in s["title"].lower()]
    others = [s for s in sections if s not in faq_secs]

    title_counts = {}
    for s in others:
        k = title_key(s["title"])
        title_counts[k] = title_counts.get(k, 0) + 1

    kept, firsts, dropped = [], {}, 0
    for s in others:
        k = title_key(s["title"])
        if s["section_id"] == "Header" or title_counts[k] < 3:
            kept.append(s)
            continue
        sig = (k, signature(s))
        if sig in firsts:                      # same template, same values -> duplicate
            firsts[sig].setdefault("repeated_in", []).append(s["section_id"])
            dropped += 1
        else:                                  # first copy, or a copy with different values
            firsts[sig] = s
            kept.append(s)

    raw_faqs = [f for s in faq_secs for f in parse_faq(s)]
    faqs = dedupe_faqs(raw_faqs)

    for rec in kept + faqs:
        rec.update({"doc_stem": stem, "doc_code": doc_code, "doc_title": doc_title})

    info = {"rupee_fixes": n_rupee, "sections_kept": len(kept), "sections_dropped": dropped,
            "faq_raw": len(raw_faqs), "faq_unique": len(faqs)}
    return kept, faqs, info


ALL_SECTIONS, ALL_FAQS = [], []
for stem, raw_text in raw.items():
    secs, faqs, info = parse_doc(stem, raw_text)
    ALL_SECTIONS += secs
    ALL_FAQS += faqs
    print(stem, info)

(PROC_DIR / "sections.json").write_text(json.dumps(ALL_SECTIONS, ensure_ascii=False, indent=1), encoding="utf-8")
(PROC_DIR / "faqs.json").write_text(json.dumps(ALL_FAQS, ensure_ascii=False, indent=1), encoding="utf-8")
print("\nTotal:", len(ALL_SECTIONS), "sections,", len(ALL_FAQS), "unique FAQs")

sample_1 {'rupee_fixes': 121, 'sections_kept': 9, 'sections_dropped': 14, 'faq_raw': 100, 'faq_unique': 10}
sample_2 {'rupee_fixes': 49, 'sections_kept': 6, 'sections_dropped': 16, 'faq_raw': 100, 'faq_unique': 10}
sample_3 {'rupee_fixes': 145, 'sections_kept': 6, 'sections_dropped': 16, 'faq_raw': 100, 'faq_unique': 10}
sample_4 {'rupee_fixes': 235, 'sections_kept': 6, 'sections_dropped': 16, 'faq_raw': 100, 'faq_unique': 10}
sample_5 {'rupee_fixes': 88, 'sections_kept': 10, 'sections_dropped': 13, 'faq_raw': 100, 'faq_unique': 10}
sample_6 {'rupee_fixes': 48, 'sections_kept': 9, 'sections_dropped': 15, 'faq_raw': 100, 'faq_unique': 10}

Total: 46 sections, 60 unique FAQs


### which sections survived in each file

In [21]:
for stem in sorted({s["doc_stem"] for s in ALL_SECTIONS}):
    print("=" * 60, stem)
    for s in ALL_SECTIONS:
        if s["doc_stem"] == stem:
            extra = f"  (repeated in {s['repeated_in'][0]}..{s['repeated_in'][-1]})" if s.get("repeated_in") else ""
            print(f"  Section {s['section_id']:<12} p{s['page']:<3} {s['title'][:55]}{extra}")

============================================================ sample_1
  Section Header       p1   Document header
  Section 1            p1   Fixed Deposit Tenures, Interest Rates, and Senior Citiz
  Section 2            p2   Premature Liquidation and Penalties
  Section 3            p2   Tax Deducted at Source (TDS) & Form 15G / 15H
  Section 4            p2   FinBase Wealth — Mutual Funds & SIP
  Section 5            p2   FinBase Wealth — 24K Digital Gold and Silver
  Section 6            p2   Wealth Portfolio Risk & Asset Management Standards 6  (repeated in 7..20)
  Section 22           p8   Explicit Product Exclusions & Unsupported Services Poli
  Section FDW-EXT-001  p25  Extended Operational Protocol 1
============================================================ sample_2
  Section Header       p1   Document header
  Section 1            p1   UPI Limits and 24-Hour Security Cooling Period
  Section 2            p1   Failed Transactions and Auto-Reversal Turnaround Time (
  Sectio

### spot checks

In [23]:
s1 = next(s for s in ALL_SECTIONS if s["doc_stem"] == "sample_1" and s["section_id"] == "1")
print(s1["text"])
print("\n" + "-" * 60 + "\n")
print(ALL_FAQS[0])

FinBase offers digital Fixed Deposits starting with a minimum deposit of ₹1,000 up to ₹50,00,000 per booking:
Tenure Bucket | Regular Customer Rate (p.a.) | Senior Citizen Rate (60+ yrs) | Compounding Frequency
7 days to 180 days | 5.50% p.a. | 6.00% p.a. | At Maturity
181 days to 364 days | 6.25% p.a. | 6.75% p.a. | Quarterly Compounded
1 year to less than 2 years | 7.50% p.a. | 8.00% p.a. | Quarterly Compounded
2 years to 3 years | 7.25% p.a. | 7.75% p.a. | Quarterly Compounded
3 years to 5 years | 7.00% p.a. | 7.50% p.a. | Quarterly Compounded
• Senior Citizen Rate Advantage: Senior citizen depositors aged 60 years and above are entitled to an additional interest rate of 0.50%
p.a. across all deposit tenures.

------------------------------------------------------------

{'faq_id': 'Q001', 'question': 'What is the interest rate for senior citizens on a 1-year Fixed Deposit?', 'answer': 'Under Section 1, senior citizens (aged 60 and above) receive an extra 0.50% p.a. over the regular

### helpers

In [25]:
import statistics
from collections import Counter

SUBHEAD = re.compile(r"^(\d+\.\d+)\s+(\S.*)$")
MAX_CHARS = 1200


def short_title(t):
    return re.sub(r"^FinBase\s+", "", t).strip()


def prep_lines(text):
    """Split glued bullets/sub-headings, then re-join hard-wrapped lines."""
    text = re.sub(r"\s*•\s*", "\n• ", text)
    text = re.sub(r"(?<=[.!?)]) +(?=\d+\.\d+ [A-Z])", "\n", text)
    lines = []
    for raw_line in text.split("\n"):
        line = raw_line.strip()
        if not line:
            continue
        if lines:
            prev = lines[-1]
            table = " | " in line or " | " in prev
            heading = bool(SUBHEAD.match(line) or SUBHEAD.match(prev))
            bullet = line.startswith("•")
            if not (table or heading or bullet):
                if line[0].islower() or not re.search(r"[.:?!;)]$", prev):
                    lines[-1] = prev + " " + line
                    continue
        lines.append(line)
    return lines


def to_blocks(lines):
    """Group consecutive table rows so a table is never cut in half."""
    blocks, table = [], []
    for ln in lines:
        if " | " in ln:
            table.append(ln)
        else:
            if table:
                blocks.append("\n".join(table))
                table = []
            blocks.append(ln)
    if table:
        blocks.append("\n".join(table))
    return blocks


def pack(blocks, max_chars=MAX_CHARS):
    """Pack blocks into chunks <= max_chars; a short last block is carried over as overlap."""
    chunks, cur = [], []
    for b in blocks:
        size = sum(len(x) + 1 for x in cur) + len(b)
        if cur and size > max_chars:
            chunks.append("\n".join(cur))
            cur = [cur[-1], b] if len(cur[-1]) < 300 else [b]
        else:
            cur.append(b)
    if cur:
        chunks.append("\n".join(cur))
    return chunks

### section and FAQ chunk builders

In [27]:
def section_chunks(sec):
    sid = str(sec["section_id"])
    lines = prep_lines(sec["text"])

    parts, cur_id, cur_title, cur = [], None, None, []
    for ln in lines:
        m = SUBHEAD.match(ln)
        if m and m.group(1).split(".")[0] == sid:
            if cur:
                parts.append((cur_id, cur_title, cur))
            cur_id = m.group(1)
            if len(m.group(2)) > 90:           # heading glued to body text
                cur_title = m.group(2)[:60].rsplit(" ", 1)[0] + "…"
                cur = [ln]
            else:
                cur_title, cur = m.group(2), []
        else:
            cur.append(ln)
    if cur:
        parts.append((cur_id, cur_title, cur))

    if len(parts) > 1 and parts[0][0] is None and len("\n".join(parts[0][2])) < 80:
        first = parts.pop(0)
        parts[0] = (parts[0][0], parts[0][1], first[2] + parts[0][2])

    short = short_title(sec["doc_title"])
    out = []
    for sub_id, sub_title, plines in parts:
        loc_id = sub_id or sid
        loc = "Document header" if sid == "Header" else f"Section {loc_id}"
        if sid == "Header":
            header = f"{short} ({sec['doc_code']}) | Document header"
        else:
            heading = f"{sec['title']} — {sub_title}" if sub_title else sec["title"]
            header = f"{short} ({sec['doc_code']}) | {loc}: {heading}"
        for k, body in enumerate(pack(to_blocks(plines))):
            out.append({
                "chunk_id": f"{sec['doc_stem']}|S{sid}|{sub_id or '-'}|{k}",
                "type": "section",
                "doc_stem": sec["doc_stem"], "doc_code": sec["doc_code"], "doc_title": sec["doc_title"],
                "section_id": sid, "sub_id": sub_id, "title": sec["title"], "page": sec["page"],
                "repeated_in": sec.get("repeated_in", []),
                "citation": f"{short} ({sec['doc_code']}) — {loc}, p.{sec['page']}",
                "text": header + "\n" + body,
            })
    return out


def faq_chunk(f):
    short = short_title(f["doc_title"])
    answer = re.sub(r"\s*\[Reference[^\]]*\]\.?", "", f["answer"]).strip()
    ref = re.search(r"Clause\s+([A-Z0-9\-]+)", f["answer"])
    src = re.search(r"Section\s+(\d+(?:\.\d+)?)", f["answer"])
    details = [re.sub(r"\s+", " ", d).strip() for d in f["details"]]

    header = f"{short} ({f['doc_code']}) | FAQ {f['faq_id']}"
    body = f"Q: {f['question']}\nA: {answer}"
    if details:
        body += "\n" + "\n".join(details)
    cite = f"{short} ({f['doc_code']}) — "
    cite += f"Section {src.group(1)}, FAQ {f['faq_id']}" if src else f"FAQ {f['faq_id']}"
    return {
        "chunk_id": f"{f['doc_stem']}|{f['faq_id']}",
        "type": "faq",
        "doc_stem": f["doc_stem"], "doc_code": f["doc_code"], "doc_title": f["doc_title"],
        "section_id": src.group(1) if src else None, "sub_id": None,
        "title": f["question"], "page": f["page"],
        "faq_id": f["faq_id"], "clause_ref": ref.group(1) if ref else None,
        "duplicate_ids": f.get("duplicate_ids", []),
        "citation": cite,
        "text": header + "\n" + body,
    }

### build and save

In [29]:
CHUNKS = []
for sec in ALL_SECTIONS:
    CHUNKS += section_chunks(sec)
for f in ALL_FAQS:
    CHUNKS.append(faq_chunk(f))

assert len({c["chunk_id"] for c in CHUNKS}) == len(CHUNKS), "duplicate chunk ids"
(PROC_DIR / "chunks.json").write_text(json.dumps(CHUNKS, ensure_ascii=False, indent=1), encoding="utf-8")

by = Counter((c["doc_stem"], c["type"]) for c in CHUNKS)
for stem in sorted({c["doc_stem"] for c in CHUNKS}):
    print(stem, "| section chunks:", by[(stem, "section")], "| faq chunks:", by[(stem, "faq")])

lens = [len(c["text"]) for c in CHUNKS]
print("\nTotal chunks:", len(CHUNKS))
print("Chars min/avg/max:", min(lens), int(statistics.mean(lens)), max(lens))

sample_1 | section chunks: 9 | faq chunks: 10
sample_2 | section chunks: 7 | faq chunks: 10
sample_3 | section chunks: 12 | faq chunks: 10
sample_4 | section chunks: 6 | faq chunks: 10
sample_5 | section chunks: 16 | faq chunks: 10
sample_6 | section chunks: 10 | faq chunks: 10

Total chunks: 120
Chars min/avg/max: 327 630 1307


### check that key facts survived

In [31]:
CHECKS = [
    ("sample_1", "8.00% p.a."), ("sample_1", "₹50,00,000"), ("sample_1", "Form 15H"),
    ("sample_1", "cryptocurrency"),
    ("sample_2", "T+2 business days"), ("sample_2", "₹100 per calendar day"), ("sample_2", "U69"),
    ("sample_3", "3.49% per month"), ("sample_3", "Priority Pass"), ("sample_3", "₹4,999"),
    ("sample_3", "Late Payment Fee"),
]
for stem, needle in CHECKS:
    hit = next((c["chunk_id"] for c in CHUNKS if c["doc_stem"] == stem and needle.lower() in c["text"].lower()), None)
    print("PASS" if hit else "FAIL", "|", stem, "|", needle, "|", hit or "-")

PASS | sample_1 | 8.00% p.a. | sample_1|S1|-|0
PASS | sample_1 | ₹50,00,000 | sample_1|S1|-|0
PASS | sample_1 | Form 15H | sample_1|S3|-|0
PASS | sample_1 | cryptocurrency | sample_1|S22|-|0
PASS | sample_2 | T+2 business days | sample_2|S2|-|0
PASS | sample_2 | ₹100 per calendar day | sample_2|S2|-|0
PASS | sample_2 | U69 | sample_2|S4|-|0
PASS | sample_3 | 3.49% per month | sample_3|S3|3.3|0
PASS | sample_3 | Priority Pass | sample_3|S1|1.3|0
PASS | sample_3 | ₹4,999 | sample_3|S1|1.3|0
PASS | sample_3 | Late Payment Fee | sample_3|S21|-|0


### look at real chunks

In [33]:
sub = next((c for c in CHUNKS if c["doc_stem"] == "sample_3" and c.get("sub_id") == "1.2"), None)
faq = next((c for c in CHUNKS if c["type"] == "faq" and c["doc_stem"] == "sample_2"), None)
for c in (sub, faq):
    if c:
        print(c["citation"])
        print(c["text"])
        print("-" * 60)
    else:
        print("chunk not found")

Credit Cards Comprehensive Handbook & Cardholder Agreement (FB-POL-CC-2026-V5) — Section 1.2, p.1
Credit Cards Comprehensive Handbook & Cardholder Agreement (FB-POL-CC-2026-V5) | Section 1.2: Introduction and Card Tiers Overview — FinBase Luxe Credit Card (Lifestyle & Travel)
• Joining & Annual Fee: ₹999 + 18% GST.
• Annual Fee Waiver: The annual fee for subsequent years is 100% waived if the total annual card spend exceeds ₹1,20,000 in the preceding membership year.
• Rewards: 2.0% cashback on online e-commerce transactions, 1.0% cashback on in-store offline POS swipes.
• Airport Lounge Access: 4 complimentary domestic airport lounge visits per calendar year (strictly capped at 1 visit per calendar quarter).
• Foreign Currency Markup Fee: Reduced markup of 1.50% on all international and foreign currency payments (industry standard is 3.50%).
------------------------------------------------------------
Payments, UPI & Refund Settlement SOP (FB-SOP-PAY-2026-V4) — Section 2, FAQ Q001
Pay

In [34]:
import sys
print("Python:", sys.version)
%pip install sentence-transformers chromadb rank-bm25

Python: 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:20:11) [MSC v.1938 64 bit (AMD64)]
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\PROJECTS\ARTH Assessment\finbase-rag\venv\Scripts\python.exe -m pip install --upgrade pip


### Load chunks

In [36]:
import json
from pathlib import Path

CHUNKS = json.loads(Path("data/processed/chunks.json").read_text(encoding="utf-8"))
CHUNK_BY_ID = {c["chunk_id"]: c for c in CHUNKS}
print("Loaded chunks:", len(CHUNKS))

Loaded chunks: 120


### Embed all chunks

In [38]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "BAAI/bge-small-en-v1.5"
model = SentenceTransformer(EMBED_MODEL)

texts = [c["text"] for c in CHUNKS]
embeddings = model.encode(texts, batch_size=32, normalize_embeddings=True, show_progress_bar=True)
print("Embeddings shape:", embeddings.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Embeddings shape: (120, 384)


### Store in ChromaDB

In [40]:
%pip install -U typing_extensions pydantic pydantic-core

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\PROJECTS\ARTH Assessment\finbase-rag\venv\Scripts\python.exe -m pip install --upgrade pip


In [41]:
%pip install --upgrade --force-reinstall --no-deps typing_extensions
%pip show typing_extensions

  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
Using cached typing_extensions-4.16.0-py3-none-any.whl (45 kB)
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.16.0
    Uninstalling typing_extensions-4.16.0:
      Successfully uninstalled typing_extensions-4.16.0
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\PROJECTS\ARTH Assessment\finbase-rag\venv\Scripts\python.exe -m pip install --upgrade pip


Name: typing_extensions
Version: 4.16.0
Summary: Backported and Experimental Type Hints for Python 3.9+
Home-page: 
Author: 
Note: you may need to restart the kernel to use updated packages.


--- Logging error ---
Traceback (most recent call last):
  File "C:\PROJECTS\ARTH Assessment\finbase-rag\venv\Lib\site-packages\pip\_internal\utils\logging.py", line 177, in emit
    self.console.print(renderable, overflow="ignore", crop=False, style=style)
  File "C:\PROJECTS\ARTH Assessment\finbase-rag\venv\Lib\site-packages\pip\_vendor\rich\console.py", line 1672, in print
    with self:
  File "C:\PROJECTS\ARTH Assessment\finbase-rag\venv\Lib\site-packages\pip\_vendor\rich\console.py", line 864, in __exit__
    self._exit_buffer()
  File "C:\PROJECTS\ARTH Assessment\finbase-rag\venv\Lib\site-packages\pip\_vendor\rich\console.py", line 822, in _exit_buffer
    self._check_buffer()
  File "C:\PROJECTS\ARTH Assessment\finbase-rag\venv\Lib\site-packages\pip\_vendor\rich\console.py", line 2027, in _check_buffer
    legacy_windows_render(buffer, LegacyWindowsTerm(self.file))
  File "C:\PROJECTS\ARTH Assessment\finbase-rag\venv\Lib\site-packages\pip\_vendor\rich\_windows_renderer.py", lin

In [42]:
from typing_extensions import Sentinel
import chromadb
print("OK, chromadb version:", chromadb.__version__)

OK, chromadb version: 1.5.9


In [43]:
import chromadb


def make_meta(c):
    return {
        "doc_stem": c["doc_stem"],
        "doc_code": c["doc_code"],
        "type": c["type"],
        "section_id": str(c.get("section_id") or ""),
        "sub_id": c.get("sub_id") or "",
        "page": int(c["page"]),
        "citation": c["citation"],
    }


client = chromadb.PersistentClient(path="data/chroma_db")
try:
    client.delete_collection("finbase")
except Exception:
    pass

collection = client.create_collection("finbase", metadata={"hnsw:space": "cosine"})
collection.add(
    ids=[c["chunk_id"] for c in CHUNKS],
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=[make_meta(c) for c in CHUNKS],
)
print("Chunks in vector store:", collection.count())

Chunks in vector store: 120


### Retrieval functions

In [45]:
import re
import numpy as np
from rank_bm25 import BM25Okapi

QUERY_PREFIX = "Represent this sentence for searching relevant passages: "


def tokenize(text):
    return re.findall(r"\w+", text.lower())


bm25 = BM25Okapi([tokenize(c["text"]) for c in CHUNKS])


def vector_search(query):
    """Cosine similarity against every chunk (only 120, so we rank them all)."""
    q = model.encode([QUERY_PREFIX + query], normalize_embeddings=True).tolist()
    res = collection.query(query_embeddings=q, n_results=len(CHUNKS))
    return [(cid, 1.0 - dist) for cid, dist in zip(res["ids"][0], res["distances"][0])]


def bm25_search(query, k=20):
    scores = bm25.get_scores(tokenize(query))
    top = np.argsort(scores)[::-1][:k]
    return [(CHUNKS[i]["chunk_id"], float(scores[i])) for i in top]


def retrieve(query, k=5, mode="hybrid"):
    """mode: 'vector', 'bm25' or 'hybrid' (reciprocal rank fusion of both)."""
    vec = vector_search(query)
    sims = dict(vec)

    if mode == "vector":
        ranked = vec[:k]
    elif mode == "bm25":
        ranked = bm25_search(query, k)
    else:
        fused = {}
        for ranking in (vec[:20], bm25_search(query, 20)):
            for rank, (cid, _) in enumerate(ranking):
                fused[cid] = fused.get(cid, 0.0) + 1.0 / (60 + rank + 1)
        ranked = sorted(fused.items(), key=lambda x: x[1], reverse=True)[:k]

    return [
        {"chunk": CHUNK_BY_ID[cid], "score": float(score), "vec_sim": float(sims[cid])}
        for cid, score in ranked
    ]

### Test helper

In [47]:
def show(query, k=3, mode="hybrid"):
    print(f"\nQ: {query}   [{mode}]")
    for r in retrieve(query, k, mode):
        c = r["chunk"]
        label = f"{c['doc_stem']} | {c['citation'].split('— ')[-1]}"
        print(f"  sim={r['vec_sim']:.3f} | {label}")

### Test queries

In [49]:
in_scope = [
    "What is the interest rate for senior citizens on a 1-year fixed deposit?",
    "How much can I send by UPI in the first 24 hours after resetting my PIN?",
    "What is the foreign currency markup on the Luxe card?",
    "I was debited for a failed transaction, how much compensation do I get if the refund is late?",
    "What is the late payment fee on a credit card?",
    "Can I buy Bitcoin on FinBase?",
    "What is the foreclosure charge on a personal loan?",
    "Which documents are accepted for KYC?",
]
out_of_scope = [
    "What reward points do I get on the Metal card?",
    "What is the capital of France?",
    "Can I get a home loan from FinBase?",
]

print("######## IN SCOPE ########")
for q in in_scope:
    show(q)

print("\n\n######## OUT OF SCOPE ########")
for q in out_of_scope:
    show(q, k=2)

######## IN SCOPE ########

Q: What is the interest rate for senior citizens on a 1-year fixed deposit?   [hybrid]
  sim=0.880 | sample_1 | Section 1, FAQ Q001
  sim=0.806 | sample_1 | Section 1, p.1
  sim=0.700 | sample_1 | Section 3, p.2

Q: How much can I send by UPI in the first 24 hours after resetting my PIN?   [hybrid]
  sim=0.843 | sample_2 | Section 1, FAQ Q003
  sim=0.774 | sample_2 | Section 1, p.1
  sim=0.724 | sample_2 | FAQ Q007

Q: What is the foreign currency markup on the Luxe card?   [hybrid]
  sim=0.857 | sample_3 | Section 1.2, FAQ Q001
  sim=0.769 | sample_3 | Section 1.3, FAQ Q006
  sim=0.764 | sample_3 | Section 1.2, p.1

Q: I was debited for a failed transaction, how much compensation do I get if the refund is late?   [hybrid]
  sim=0.767 | sample_2 | Section 2, p.1
  sim=0.773 | sample_2 | Section 2, FAQ Q002
  sim=0.764 | sample_2 | Section 2, FAQ Q001

Q: What is the late payment fee on a credit card?   [hybrid]
  sim=0.745 | sample_3 | Section 21, p.9
  sim=

### Compare the three modes

q = "What is the cash advance fee on a credit card?"
for mode in ("vector", "bm25", "hybrid"):
    show(q, k=3, mode=mode)

### Check foreclosure chunks

In [53]:
for r in retrieve("What is the foreclosure charge on a personal loan?", k=3):
    print(r["chunk"]["citation"])
    print(r["chunk"]["text"])
    print("-" * 60)

Personal Loans Master Policy & Operational Manual (FB-POL-PL-2026-V4) — Section 4.2, FAQ Q001
Personal Loans Master Policy & Operational Manual (FB-POL-PL-2026-V4) | FAQ Q001
Q: What is the foreclosure charge if I close my personal loan in 18 months?
A: According to FinBase Personal Loan Policy Section 4.2, because the loan is closed prior to completing 24 months, the applicable foreclosure charge is 3% of the outstanding principal balance plus 18% GST.
• Applicability: All retail borrowers under loan tier code PL-02.
• Verification note: Checked automatically via core lending system rule PL-VAL-0001.
• Exception criteria: Any deviation requires approval from Risk Level 2 Underwriting Manager.
• Escalation SLA: 24 business hours for credit review.
------------------------------------------------------------
Personal Loans Master Policy & Operational Manual (FB-POL-PL-2026-V4) — Section 4.2, FAQ Q002
Personal Loans Master Policy & Operational Manual (FB-POL-PL-2026-V4) | FAQ Q002
Q: Wha

In [54]:
import os, json, re, time
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv(dotenv_path=Path.cwd() / ".env", override=True)
API_KEY = (os.getenv("GEMINI_API_KEY") or "").strip().strip('"').strip("'")
assert API_KEY, "GEMINI_API_KEY not found. Check that .env is in the finbase-rag folder (not .env.txt) and has the line GEMINI_API_KEY=..."

client = genai.Client(api_key=API_KEY)
print("Key loaded, length:", len(API_KEY))

Key loaded, length: 53


### Find working models

In [56]:
CANDIDATES = [
    "gemini-3.1-flash-lite", "gemini-flash-lite-latest",
    "gemini-3.5-flash-lite", "gemini-flash-latest", "gemma-4-31b-it",
]

WORKING_MODELS = []
print("Testing candidates:")
for name in CANDIDATES:
    for attempt in range(2):
        try:
            r = client.models.generate_content(model=name, contents="Reply with the single word: ready")
            print(f"  OK   {name} -> {r.text.strip()[:30]}")
            WORKING_MODELS.append(name)
            break
        except Exception as e:
            msg = str(e)
            if "503" in msg and attempt == 0:
                time.sleep(4)
                continue
            print(f"  FAIL {name} -> {msg[:110]}")
            break

assert WORKING_MODELS, "No model worked. Paste the FAIL lines above and I'll fix it."
print("\nWill use, in order:", WORKING_MODELS)

Testing candidates:
  OK   gemini-3.1-flash-lite -> ready
  OK   gemini-flash-lite-latest -> ready
  OK   gemini-3.5-flash-lite -> ready
  OK   gemini-flash-latest -> ready
  FAIL gemma-4-31b-it -> 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}

Will use, in order: ['gemini-3.1-flash-lite', 'gemini-flash-lite-latest', 'gemini-3.5-flash-lite', 'gemini-flash-latest']


### LLM helpers and prompts

In [58]:
NOT_FOUND_MSG = ("I couldn't find this in the FinBase knowledge base. "
                 "Please contact FinBase customer support for help with this.")


def call_llm(prompt, temperature=0.0):
    last_err = None
    for model_name in WORKING_MODELS:
        for attempt in range(2):
            try:
                r = client.models.generate_content(
                    model=model_name, contents=prompt,
                    config=types.GenerateContentConfig(temperature=temperature),
                )
                return (r.text or ""), model_name
            except Exception as e:
                last_err = e
                time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"All models failed: {str(last_err)[:200]}")


def rewrite_query(question, history):
    """Turn a follow-up like 'and for Metal?' into a standalone question."""
    if not history:
        return question
    convo = "\n".join(f"User: {h['q']}\nAssistant: {h['a'][:300]}" for h in history[-3:])
    prompt = (
        "Rewrite the user's latest question as one standalone question that makes sense "
        "without the conversation. Keep all product names, amounts and terms. "
        "If it is already standalone, return it unchanged. Output only the question.\n\n"
        f"Conversation:\n{convo}\n\nLatest question: {question}\n\nStandalone question:"
    )
    text, _ = call_llm(prompt)
    return text.strip().strip('"') or question


PROMPT_TEMPLATE = """You are FinBase's customer support assistant. Answer ONLY from the numbered context passages below.

Rules:
1. Use only facts that appear in the context. Never use outside knowledge and never guess.
2. If the context does not contain the answer, set "found" to false.
3. If the context says FinBase does not offer something, say it is not offered and cite it (found = true).
4. If passages give different values for the same thing (for example for different transaction types), state each value and the section it comes from.
5. Quote amounts, percentages and timelines exactly as written, keeping the ₹ symbol.
6. Keep the answer short and in plain language (2-4 sentences). Give no personal financial advice.
7. In "sources", list the numbers of the passages you actually used.
8. Never write passage numbers such as [1] or "source 3" inside the answer text. If you mention where something is stated, use the section name from the passage label, for example "Section 1.3".

Return ONLY valid JSON, with no other text:
{{"found": true or false, "answer": "your answer", "sources": [1, 2]}}

Context:
{context}

Question: {question}
"""


def build_prompt(question, hits):
    blocks = [f"[{i}] ({h['chunk']['citation']})\n{h['chunk']['text']}" for i, h in enumerate(hits, 1)]
    return PROMPT_TEMPLATE.format(context="\n\n".join(blocks), question=question)


def parse_json(text):
    t = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
    m = re.search(r"\{.*\}", t, flags=re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

### 4b - Rank models by speed

In [60]:
test_q = "What is the foreign currency markup on the Luxe card?"
test_prompt = build_prompt(test_q, retrieve(test_q, k=5))

timings = []
for name in WORKING_MODELS:
    t0 = time.time()
    try:
        r = client.models.generate_content(
            model=name, contents=test_prompt,
            config=types.GenerateContentConfig(temperature=0),
        )
        ok = parse_json(r.text or "") is not None
        timings.append((time.time() - t0, name, ok))
    except Exception as e:
        timings.append((999.0, name, False))
        print(f"FAIL {name} -> {str(e)[:100]}")

for secs, name, ok in sorted(timings):
    print(f"{secs:6.1f}s | valid JSON={ok} | {name}")

WORKING_MODELS = [name for secs, name, ok in sorted(timings) if ok]
assert WORKING_MODELS, "No model returned valid JSON. Paste the output above."
print("\nWill use, fastest first:", WORKING_MODELS)

   1.3s | valid JSON=True | gemini-flash-lite-latest
   1.3s | valid JSON=True | gemini-3.5-flash-lite
   8.0s | valid JSON=True | gemini-3.1-flash-lite
  13.1s | valid JSON=True | gemini-flash-latest

Will use, fastest first: ['gemini-flash-lite-latest', 'gemini-3.5-flash-lite', 'gemini-3.1-flash-lite', 'gemini-flash-latest']


### 4c Updated prompt

In [122]:
PROMPT_TEMPLATE = """You are FinBase's customer support assistant. Answer ONLY from the numbered context passages below.

Rules:
1. Use only facts that appear in the context. Never use outside knowledge and never guess.
2. If the context does not contain the answer, set "found" to false.
3. If the context says FinBase does not offer something, say it is not offered and cite it (found = true).
4. If passages give different values for the same thing (for example for different transaction types), state each value and the section it comes from.
5. Quote amounts, percentages and timelines exactly as written, keeping the ₹ symbol.
6. Keep the answer short and in plain language (2-4 sentences). Give no personal financial advice.
7. In "sources", list the numbers of the passages you actually used.
8. Never write passage numbers such as [1] or "source 3" inside the answer text. When you say where something is stated, name the document and section from the passage label, for example "Payments SOP, Section 1".
9. Numbered sections are the main policy text and FAQ passages are summaries of them. If an FAQ passage differs from a numbered section, follow the numbered section and briefly note the difference.
10. Passages can come from different FinBase documents (fixed deposits, payments, credit cards, savings, loans, KYC). Never merge facts from different documents into one statement. If the question could apply to more than one document, answer for each one and name the document.

Return ONLY valid JSON, with no other text:
{{"found": true or false, "answer": "your answer", "sources": [1, 2]}}

Context:
{context}

Question: {question}
"""
print("Prompt updated.")

Prompt updated.


### Main answer function

In [62]:
MIN_SIM = 0.55   # below this the question is treated as unrelated (tuned later)
TOP_K = 5


def answer_question(question, history=None, k=TOP_K, mode="hybrid"):
    t0 = time.time()
    standalone = rewrite_query(question, history or [])
    hits = retrieve(standalone, k=k, mode=mode)
    top_sim = max(h["vec_sim"] for h in hits)

    result = {"question": question, "standalone": standalone, "top_sim": top_sim,
              "retrieved": hits, "model": None, "found": False, "answer": NOT_FOUND_MSG,
              "sources": [], "reason": "", "seconds": 0.0}

    if top_sim < MIN_SIM:
        result["reason"] = "low_similarity"
        result["seconds"] = time.time() - t0
        return result

    prompt = build_prompt(standalone, hits)
    data = None
    for _ in range(2):
        text, model_name = call_llm(prompt)
        data = parse_json(text)
        if data and "answer" in data:
            break
    result["model"] = model_name

    if not data or "answer" not in data:
        result["reason"] = "bad_llm_output"
    elif not data.get("found", False):
        result["reason"] = "llm_not_found"
    else:
        idx = [i for i in data.get("sources", []) if isinstance(i, int) and 1 <= i <= len(hits)]
        idx = list(dict.fromkeys(idx))
        if not idx:
            idx = [1]
            result["reason"] = "source_fallback"
        result.update({"found": True, "answer": str(data["answer"]).strip(),
                       "sources": [hits[i - 1]["chunk"] for i in idx]})

    result["seconds"] = time.time() - t0
    return result

### Chat helper

In [64]:
HISTORY = []


def ask(question, use_history=False, k=TOP_K, return_result=False):
    res = answer_question(question, HISTORY if use_history else [], k=k)
    if use_history:
        HISTORY.append({"q": question, "a": res["answer"]})

    print(f"\nQ: {question}")
    if res["standalone"] != question:
        print(f"   (searched as: {res['standalone']})")
    print(f"A: {res['answer']}")
    if res["sources"]:
        print("Sources:")
        for c in res["sources"]:
            print("  -", c["citation"])
    print(f"[found={res['found']} | top_sim={res['top_sim']:.2f} | reason={res['reason'] or 'ok'} "
          f"| model={res['model']} | {res['seconds']:.1f}s]")
    return res if return_result else None

### Test: in-scope questions

In [66]:
for q in [
    "What is the interest rate for senior citizens on a 1-year fixed deposit?",
    "I was debited for a failed transaction. How much compensation do I get if the refund is late?",
    "What is the foreclosure charge on a personal loan?",
    "What is the late payment fee on a credit card?",
]:
    ask(q)


Q: What is the interest rate for senior citizens on a 1-year fixed deposit?
A: Under Section 1, senior citizens aged 60 and above receive an extra 0.50% p.a. over the regular rate of 7.50% p.a., earning an effective rate of 8.00% p.a. on a 1 to 2-year Fixed Deposit.
Sources:
  - Fixed Deposits & Wealth Products Master Guidelines (FB-POL-FDW-2026-V3) — Section 1, FAQ Q001
  - Fixed Deposits & Wealth Products Master Guidelines (FB-POL-FDW-2026-V3) — Section 1, p.1
[found=True | top_sim=0.88 | reason=ok | model=gemini-flash-lite-latest | 1.5s]

Q: I was debited for a failed transaction. How much compensation do I get if the refund is late?
A: For delayed refunds on failed transactions, FinBase compensates ₹100 per calendar day of delay, as stated in Section 2 and Section 2.1. Note that the matrix in Section 21 specifies ₹100 per day beyond T+2 for P2P UPI Failed Debit, ₹100 per day beyond T+5 for P2M Merchant Online Debit Failed, ATM Cash Not Dispensed, and POS Card Transaction Failed, ₹

### Test: not-found cases and follow-ups

In [68]:
print("######## NOT IN KNOWLEDGE BASE ########")
for q in [
    "How do I redeem my credit card reward points?",
    "Can I get a home loan from FinBase?",
    "What is the capital of France?",
]:
    ask(q)

print("\n\n######## IN KB: OFFERED / NOT OFFERED ########")
for q in [
    "What reward rate does the Metal card give?",
    "Does FinBase offer cryptocurrency trading?",
]:
    ask(q)

print("\n\n######## FOLLOW-UPS ########")
HISTORY.clear()
ask("What is the foreign currency markup on the Luxe card?", use_history=True)
ask("And on the Metal card?", use_history=True)
ask("Which one is cheaper for international spends?", use_history=True)

######## NOT IN KNOWLEDGE BASE ########

Q: How do I redeem my credit card reward points?
A: I couldn't find this in the FinBase knowledge base. Please contact FinBase customer support for help with this.
[found=False | top_sim=0.59 | reason=llm_not_found | model=gemini-flash-lite-latest | 1.1s]

Q: Can I get a home loan from FinBase?
A: I couldn't find this in the FinBase knowledge base. Please contact FinBase customer support for help with this.
[found=False | top_sim=0.76 | reason=llm_not_found | model=gemini-flash-lite-latest | 1.4s]

Q: What is the capital of France?
A: I couldn't find this in the FinBase knowledge base. Please contact FinBase customer support for help with this.
[found=False | top_sim=0.45 | reason=low_similarity | model=None | 0.1s]


######## IN KB: OFFERED / NOT OFFERED ########

Q: What reward rate does the Metal card give?
A: As stated in Section 1.3, the FinBase Metal card offers a reward rate of 3.5% reward points on flights, hotels, fine dining, and inter

### Evaluation set

In [70]:
EVAL_SET = [
    # ---------- sample_1: Fixed Deposits & Wealth ----------
    {"id": "E01", "type": "answerable", "question": "I'm 65. What interest rate will I get on an 18-month fixed deposit?",
     "expected": ["sample_1|S1|", "sample_1|Q001"], "facts": ["8.00%"]},
    {"id": "E02", "type": "answerable", "question": "What is the smallest amount I can put into a fixed deposit?",
     "expected": ["sample_1|S1|"], "facts": ["1,000"]},
    {"id": "E03", "type": "answerable", "question": "How much penalty do I pay if I break my FD before maturity?",
     "expected": ["sample_1|S2|", "sample_1|Q002"], "facts": ["1.00%"]},
    {"id": "E04", "type": "answerable", "question": "If I close my FD within a week of booking, how much interest do I earn?",
     "expected": ["sample_1|S2|", "sample_1|Q002"], "facts": ["zero|no interest"]},
    {"id": "E05", "type": "answerable", "question": "What TDS rate applies if my PAN is not linked to my account?",
     "expected": ["sample_1|S3|"], "facts": ["20%"]},
    {"id": "E06", "type": "answerable", "question": "Above what yearly interest income is TDS deducted for senior citizens?",
     "expected": ["sample_1|S3|"], "facts": ["50,000"]},
    {"id": "E07", "type": "answerable", "question": "What's the minimum monthly amount to start a mutual fund SIP?",
     "expected": ["sample_1|S4|", "sample_1|Q004"], "facts": ["₹100|rs.100|rs100|100permonth"]},
    {"id": "E08", "type": "answerable", "question": "Can I trade Bitcoin or Ethereum on FinBase?",
     "expected": ["sample_1|S22|", "sample_1|Q006"], "facts": ["not"]},
    {"id": "E09", "type": "answerable", "question": "Can I get a loan against my farmland?",
     "expected": ["sample_1|S22|", "sample_1|Q007"], "facts": ["not"]},
    {"id": "E10", "type": "answerable", "question": "Do you provide daily intraday stock tips?",
     "expected": ["sample_1|S22|", "sample_1|Q008"], "facts": ["not"]},
    {"id": "E11", "type": "answerable", "question": "What's the maximum I can invest in a single FD booking?",
     "expected": ["sample_1|S1|", "sample_1|Q009"], "facts": ["50,00,000"]},
    {"id": "E12", "type": "answerable", "question": "Are my FinBase deposits insured, and up to how much?",
     "expected": ["sample_1|Q010"], "facts": ["5,00,000"]},
    {"id": "E13", "type": "answerable", "question": "What is the smallest purchase I can make in digital gold?",
     "expected": ["sample_1|S5|", "sample_1|Q005"], "facts": ["₹10"]},

    # ---------- sample_2: Payments, UPI & Refunds ----------
    {"id": "E14", "type": "answerable", "question": "How much can I transfer through UPI in one day?",
     "expected": ["sample_2|S1|", "sample_2|Q004"], "facts": ["1,00,000"]},
    {"id": "E15", "type": "answerable", "question": "I changed my UPI PIN today. What's the most I can send in the next 24 hours?",
     "expected": ["sample_2|S1|", "sample_2|Q003"], "facts": ["5,000"]},
    {"id": "E16", "type": "answerable", "question": "By when must I report an unauthorised debit?",
     "expected": ["sample_2|S3|", "sample_2|Q005"], "facts": ["3 calendar days|3 days"]},
    {"id": "E17", "type": "answerable", "question": "How quickly do I get temporary credit for a disputed merchant transaction?",
     "expected": ["sample_2|S3|", "sample_2|Q006"], "facts": ["7 working days|7 business days"]},
    {"id": "E18", "type": "answerable", "question": "Is there any charge for sending money through UPI?",
     "expected": ["sample_2|Q009"], "facts": ["not|no fee|free|no charge"]},
    {"id": "E19", "type": "answerable", "question": "What does the U69 error mean?",
     "expected": ["sample_2|Q008", "sample_2|S4|"], "facts": ["timeout"]},
    {"id": "E20", "type": "multi", "question": "My UPI payment failed with U69 but the money left my account. When is it refunded, and what do I get if it's late?",
     "expected": ["sample_2|S2|", "sample_2|Q001", "sample_2|Q002", "sample_2|Q008"],
     "facts": ["t+2", "100 per calendar day|100 per day|₹100"]},
    {"id": "E21", "type": "multi", "question": "What compensation do I get if ATM cash was not dispensed and the refund is delayed?",
     "expected": ["sample_2|S21|"], "facts": ["t+5"]},

    # ---------- sample_3: Credit Cards ----------
    {"id": "E22", "type": "multi", "question": "What does the Luxe card cost per year and how can I get the fee waived?",
     "expected": ["sample_3|S1|1.2", "sample_3|Q002"], "facts": ["999", "1,20,000"]},
    {"id": "E23", "type": "answerable", "question": "How many free airport lounge visits does the Luxe card include?",
     "expected": ["sample_3|S1|1.2", "sample_3|Q004"], "facts": ["quarter"]},
    {"id": "E24", "type": "answerable", "question": "How do you calculate the minimum amount due on my card?",
     "expected": ["sample_3|S3|3.2", "sample_3|Q008"], "facts": ["5%", "250"]},
    {"id": "E25", "type": "answerable", "question": "What interest do I pay if I don't clear my card bill in full?",
     "expected": ["sample_3|S3|3.3", "sample_3|Q005"], "facts": ["3.49%"]},
    {"id": "E26", "type": "answerable", "question": "What does it cost to withdraw cash from an ATM using the credit card?",
     "expected": ["sample_3|S21|", "sample_3|Q009"], "facts": ["2.5%", "300"]},
    {"id": "E27", "type": "answerable", "question": "Does the Neo card have a yearly fee?",
     "expected": ["sample_3|S1|1.1", "sample_3|Q003"], "facts": ["free"]},
    {"id": "E28", "type": "answerable", "question": "How can I block my card right away if I lose it?",
     "expected": ["sample_3|Q007"], "facts": ["controls"]},
    {"id": "E29", "type": "answerable", "question": "How often is my credit limit considered for an increase?",
     "expected": ["sample_3|S2|", "sample_3|Q010"], "facts": ["6 months"]},
    {"id": "E30", "type": "multi", "question": "How do the Luxe and Metal cards compare on foreign currency markup?",
     "expected": ["sample_3|S1|1.2", "sample_3|S1|1.3", "sample_3|Q001", "sample_3|Q006"], "facts": ["1.50%", "1.00%"]},

    # ---------- NOT in the knowledge base ----------
    {"id": "E31", "type": "unanswerable", "question": "How do I redeem my credit card reward points?", "expected": [], "facts": []},
    {"id": "E32", "type": "unanswerable", "question": "Can I get a home loan from FinBase?", "expected": [], "facts": []},
    {"id": "E33", "type": "unanswerable", "question": "What is the capital of France?", "expected": [], "facts": []},
    {"id": "E34", "type": "unanswerable", "question": "Who is the CEO of FinBase?", "expected": [], "facts": []},
    {"id": "E35", "type": "unanswerable", "question": "Does FinBase offer travel insurance?", "expected": [], "facts": []},
]

Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("data/processed/eval_set.json").write_text(json.dumps(EVAL_SET, ensure_ascii=False, indent=1), encoding="utf-8")
print("Questions:", len(EVAL_SET), "| types:", {t: sum(e["type"] == t for e in EVAL_SET) for t in ("answerable", "multi", "unanswerable")})

Questions: 35 | types: {'answerable': 26, 'multi': 4, 'unanswerable': 5}


### Helpers and retrieval evaluation

In [134]:
import pandas as pd


def norm(s):
    return re.sub(r"\s+", "", s.lower())


def is_hit(chunk_id, expected):
    return any(chunk_id.startswith(p) for p in expected)


def facts_ok(answer, facts):
    a = norm(answer)
    return all(any(norm(alt) in a for alt in f.split("|")) for f in facts)


def retrieval_eval(mode, k=5):
    ranks = []
    for item in EVAL_SET:
        if item["type"] == "unanswerable":
            continue
        ids = [r["chunk"]["chunk_id"] for r in retrieve(item["question"], k=k, mode=mode)]
        rank = next((i + 1 for i, cid in enumerate(ids) if is_hit(cid, item["expected"])), None)
        ranks.append({"id": item["id"], "rank": rank})
    r = pd.DataFrame(ranks)["rank"].astype(float)
    return {
        "mode": mode,
        "Hit@1": round(float((r <= 1).mean()), 3),
        "Hit@3": round(float((r <= 3).mean()), 3),
        "Hit@5": round(float((r <= 5).mean()), 3),
        "MRR": round(float((1 / r).fillna(0).mean()), 3),
    }, [x["id"] for x in ranks if x["rank"] is None]


rows, misses = [], {}
for mode in ("vector", "bm25", "hybrid"):
    row, miss = retrieval_eval(mode)
    rows.append(row)
    misses[mode] = miss

print(pd.DataFrame(rows).to_string(index=False))
print("\nQuestions with no correct source in top 5:")
for mode, ids in misses.items():
    print(f"  {mode:<7}: {ids if ids else 'none'}")

  mode  Hit@1  Hit@3  Hit@5   MRR
vector  0.917  1.000  1.000 0.953
  bm25  0.767  0.867  0.883 0.815
hybrid  0.883  0.950  0.983 0.919

Questions with no correct source in top 5:
  vector : none
  bm25   : ['E02', 'E14', 'E25', 'E29', 'E45', 'E53', 'E54']
  hybrid : ['E54']


### End-to-end evaluation

In [135]:
JUDGE_PROMPT = """You are a strict fact-checker. Decide whether every claim in the ANSWER is supported by the CONTEXT.

CONTEXT:
{context}

ANSWER:
{answer}

Return ONLY JSON: {{"grounded": true or false, "reason": "one short sentence"}}"""


def judge_grounded(res):
    context = "\n\n".join(h["chunk"]["text"] for h in res["retrieved"])
    text, _ = call_llm(JUDGE_PROMPT.format(context=context, answer=res["answer"]))
    data = parse_json(text)
    return bool(data and data.get("grounded") is True)


rows = []
for item in EVAL_SET:
    row = {"id": item["id"], "type": item["type"], "question": item["question"],
           "found": False, "reason": "", "top_sim": 0.0, "answer": "", "sources": "",
           "refusal_ok": False, "answered": False, "fact_ok": False, "cite_ok": False,
           "grounded": False, "error": ""}
    try:
        res = answer_question(item["question"])
        row.update({"found": res["found"], "reason": res["reason"], "top_sim": round(res["top_sim"], 3),
                    "answer": res["answer"],
                    "sources": " ; ".join(c["citation"].split("— ")[-1] for c in res["sources"])})
        if item["type"] == "unanswerable":
            row["refusal_ok"] = not res["found"]
        else:
            row["answered"] = res["found"]
            row["fact_ok"] = res["found"] and facts_ok(res["answer"], item["facts"])
            row["cite_ok"] = any(is_hit(c["chunk_id"], item["expected"]) for c in res["sources"])
            row["grounded"] = judge_grounded(res) if res["found"] else False
    except Exception as e:
        row["error"] = str(e)[:120]
    rows.append(row)
    time.sleep(1)

df = pd.DataFrame(rows)
df.to_csv("data/eval_results.csv", index=False)

a = df[df.type != "unanswerable"]
u = df[df.type == "unanswerable"]
print(f"Answerable questions: {len(a)}")
print(f"  Answered (not refused): {a.answered.mean():.0%}")
print(f"  Fact correctness      : {a.fact_ok.mean():.0%}")
print(f"  Citation accuracy     : {a.cite_ok.mean():.0%}")
print(f"  Groundedness (judge)  : {a.grounded.mean():.0%}")
print(f"Not-in-KB questions: {len(u)}")
print(f"  Correctly refused     : {u.refusal_ok.mean():.0%}")
print(f"Errors: {int((df.error != '').sum())}")

bad_a = a[~(a.answered & a.fact_ok & a.cite_ok & a.grounded)]
bad_u = u[~u.refusal_ok]
print("\n--- FAILURES ---")
for _, r in pd.concat([bad_a, bad_u]).iterrows():
    print(f"\n{r['id']} | {r['question']}")
    print(f"  answer : {r['answer'][:220]}")
    print(f"  sources: {r['sources']}")
    print(f"  flags  : answered={r['answered']} fact={r['fact_ok']} cite={r['cite_ok']} grounded={r['grounded']} "
          f"refusal_ok={r['refusal_ok']} reason={r['reason']} {r['error']}")

Answerable questions: 60
  Answered (not refused): 98%
  Fact correctness      : 93%
  Citation accuracy     : 98%
  Groundedness (judge)  : 98%
Not-in-KB questions: 5
  Correctly refused     : 100%
Errors: 0

--- FAILURES ---

E39 | How many free atm withdrawals do I get a month
  answer : For Digital Savings Accounts, you receive 5 free cash withdrawals per month at FinBase partner ATMs, as stated in the Digital Savings Accounts & Banking Operations Manual, Section 3. For non-partner ATMs, you get 3 free 
  sources: Section 3, FAQ Q004 ; Section 3, p.2 ; Section 21, p.8
  flags  : answered=True fact=False cite=True grounded=True refusal_ok=False reason= 

E54 | How long do I have to pay back the loan?
  answer : I couldn't find this in the FinBase knowledge base. Please contact FinBase customer support for help with this.
  sources: 
  flags  : answered=False fact=False cite=False grounded=False refusal_ok=False reason=llm_not_found 

E59 | Am I liable for anything if I report fraud 

### Inspect the two failures

In [119]:
for qid in ("E14", "E22"):
    item = next(e for e in EVAL_SET if e["id"] == qid)
    res = answer_question(item["question"])
    print("=" * 70)
    print(qid, "|", item["question"])
    print("\nANSWER:", res["answer"])
    print("\nRETRIEVED:")
    for h in res["retrieved"]:
        print(f"  sim={h['vec_sim']:.2f} | {h['chunk']['citation']}")
    context = "\n\n".join(h["chunk"]["text"] for h in res["retrieved"])
    verdict, _ = call_llm(JUDGE_PROMPT.format(context=context, answer=res["answer"]))
    print("\nJUDGE:", verdict.strip()[:400])

E14 | How much can I transfer through UPI in one day?

ANSWER: The standard daily UPI limit is up to ₹1,00,000 per calendar day across peer-to-peer transfers, as stated in Section 3 and Section 1. Additionally, Section 1 and Section 3 note that outward UPI payments are restricted to a maximum cumulative limit of ₹5,000 for the first 24 hours following a PIN reset due to a security cooling period.

RETRIEVED:
  sim=0.77 | Digital Savings Accounts & Banking Operations Manual (FB-POL-SAV-2026-V3) — Section 3, p.2
  sim=0.68 | Payments, UPI & Refund Settlement SOP (FB-SOP-PAY-2026-V4) — Section 3, FAQ Q005
  sim=0.75 | Payments, UPI & Refund Settlement SOP (FB-SOP-PAY-2026-V4) — Section 1, FAQ Q003
  sim=0.72 | Payments, UPI & Refund Settlement SOP (FB-SOP-PAY-2026-V4) — Section 21, p.8
  sim=0.77 | Payments, UPI & Refund Settlement SOP (FB-SOP-PAY-2026-V4) — Section 1, p.1

JUDGE: {"grounded": false, "reason": "Section 3 does not mention the ₹5,000 cooling period limit after a PIN reset."

### Threshold check

In [136]:
df = pd.read_csv("data/eval_results.csv")

print(df.groupby("type")["top_sim"].agg(["min", "median", "max"]).round(3))

print("\nLowest-scoring answerable questions:")
print(df[df.type != "unanswerable"].sort_values("top_sim").head(5)[["id", "top_sim"]].to_string(index=False))

print("\nNot-in-KB questions, highest first:")
print(df[df.type == "unanswerable"].sort_values("top_sim", ascending=False)[["id", "top_sim", "reason"]].to_string(index=False))

                min  median    max
type                              
answerable    0.687   0.778  0.902
multi         0.748   0.793  0.827
unanswerable  0.453   0.690  0.761

Lowest-scoring answerable questions:
 id  top_sim
E65    0.687
E19    0.694
E25    0.705
E28    0.709
E04    0.710

Not-in-KB questions, highest first:
 id  top_sim         reason
E32    0.761  llm_not_found
E35    0.718  llm_not_found
E34    0.690  llm_not_found
E31    0.590  llm_not_found
E33    0.453 low_similarity


### questions for documents 4–6

In [132]:
PARA_PROMPT = ("Rewrite this customer question in different words, the way a real customer would type it. "
               "Keep the meaning and any product names. Output only the question.\n\nQuestion: {q}")
NUM = re.compile(r"₹\s?[\d,]+(?:\.\d+)?|\d+(?:\.\d+)?\s?%")

extra, n = [], 36
for c in CHUNKS:
    if c["type"] != "faq" or c["doc_stem"] not in ("sample_4", "sample_5", "sample_6"):
        continue
    q = re.search(r"^Q: (.*)$", c["text"], flags=re.M).group(1).strip()
    a = re.search(r"^A: (.*)$", c["text"], flags=re.M).group(1).strip()
    para, _ = call_llm(PARA_PROMPT.format(q=q))

    sec = c.get("section_id")
    expected = [c["chunk_id"]]
    if sec:
        major = sec.split(".")[0]
        expected.append(f"{c['doc_stem']}|S{major}|{sec}" if "." in sec else f"{c['doc_stem']}|S{sec}|")
    facts = [f.replace(" ", "") for f in NUM.findall(re.sub(r"Section\s+[\d\.]+", "", a))[:2]]

    extra.append({"id": f"E{n}", "type": "answerable", "question": para.strip().strip('"'),
                  "expected": expected, "facts": facts, "original_question": q})
    n += 1
    time.sleep(0.5)

EVAL_SET = [e for e in EVAL_SET if int(e["id"][1:]) < 36] + extra
Path("data/processed/eval_set.json").write_text(json.dumps(EVAL_SET, ensure_ascii=False, indent=1), encoding="utf-8")
print("Total questions:", len(EVAL_SET), "| added:", len(extra))
for e in extra[:6]:
    print(f"\n{e['id']} original: {e['original_question']}\n     rewritten: {e['question']}\n     facts: {e['facts']}")

Total questions: 65 | added: 30

E36 original: What is the minimum balance required for a FinBase Digital Savings Account?
     rewritten: Whats the minimum balance I need to keep in a FinBase Digital Savings Account?
     facts: []

E37 original: What interest rate do I get if my account balance is ₹5,00,000?
     rewritten: what interest rate will i get for keeping 5 lakhs in my account
     facts: ['₹1,00,000', '3.50%']

E38 original: What is the maximum interest rate offered on savings accounts?
     rewritten: What’s the highest interest rate you guys offer on savings accounts?
     facts: ['₹10,00,000', '7.00%']

E39 original: How many free ATM withdrawals are available each month?
     rewritten: How many free atm withdrawals do I get a month
     facts: ['₹21']

E40 original: Are IMPS or NEFT fund transfers chargeable?
     rewritten: Are there any charges for IMPS or NEFT transfers?
     facts: ['100%']

E41 original: What is the daily transaction limit for UPI payments?
     

In [144]:
for qid in ["E39", "E54", "E59", "E62"]:
    item = next(e for e in EVAL_SET if e["id"] == qid)
    res = answer_question(item["question"])
    faq = CHUNK_BY_ID.get(item["expected"][0])
    m = re.search(r"^A: (.*)$", faq["text"], flags=re.M) if faq else None
    print("=" * 70)
    print(qid, "| asked           :", item["question"])
    print("     original FAQ q  :", item.get("original_question"))
    print("     FAQ answer      :", m.group(1) if m else "-")
    print("     expected facts  :", item["facts"])
    print("     our answer      :", res["answer"])
    print("     our sources     :", [c["citation"].split("— ")[-1] for c in res["sources"]])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-10-09 12:28:37,027 AFC is enabled with max remote calls: 10.
2026-10-09 12:28:39,163 HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-lite-latest:generateContent "HTTP/1.1 200 OK"


E39 | asked           : How many free atm withdrawals do I get a month
     original FAQ q  : How many free ATM withdrawals are available each month?
     FAQ answer      : Under Section 3, customers receive 5 free cash withdrawals per month at partner ATMs (3 free per month at other metro ATMs). Subsequent withdrawals cost ₹21 + GST.
     expected facts  : ['₹21']
     our answer      : For Digital Savings Accounts, according to Digital Savings Accounts & Banking Operations Manual (FB-POL-SAV-2026-V3), Section 3, you receive 5 free cash withdrawals per calendar month at partner ATMs. For non-partner ATMs, you get 3 free withdrawals per month in metro cities (5 in non-metro locations). Note that the FAQ Q004 mentions 3 free per month at other metro ATMs without specifying the non-metro limit.
     our sources     : ['Section 3, FAQ Q004', 'Section 3, p.2', 'Section 21, p.8']


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-10-09 12:28:39,304 AFC is enabled with max remote calls: 10.
2026-10-09 12:28:40,429 HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-lite-latest:generateContent "HTTP/1.1 200 OK"


...

2026-10-09 12:28:40,575 AFC is enabled with max remote calls: 10.
2026-10-09 12:28:41,834 HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-lite-latest:generateContent "HTTP/1.1 200 OK"


E59 | asked           : Am I liable for anything if I report fraud after 3 days?
     original FAQ q  : What is the customer liability policy if fraud is reported within 3 days?
     FAQ answer      : Under Section 5, customers have Zero Liability (100% refund) if third-party electronic fraud is reported within 3 calendar days of receiving the transaction alert.
     expected facts  : ['100%']
     our answer      : As stated in the KYC Verification, Compliance & Security Master Document, Section 5, if fraud is reported between 4 to 7 calendar days, your liability is capped at ₹5,000 (or the transaction value, whichever is lower). Beyond 7 days, your liability is determined based on a bank board-approved dispute review and case investigation.
     our sources     : ['Section 5, p.2']


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-10-09 12:28:41,971 AFC is enabled with max remote calls: 10.
2026-10-09 12:28:43,236 HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-lite-latest:generateContent "HTTP/1.1 200 OK"


E62 | asked           : what's the max transaction limit for a minimum kyc account?
     original FAQ q  : What is the transaction limit on a Minimum KYC account?
     FAQ answer      : Under Section 1, Minimum KYC accounts have a monthly credit limit of ₹10,000 and an annual limit of ₹1,00,000, valid for 12 months.
     expected facts  : ['₹10,000', '₹1,00,000,']
     our answer      : As stated in the KYC Verification, Compliance & Security Master Document, Section 1, a Minimum KYC account has a monthly credit transaction limit of ₹10,000 and an annual cumulative limit of ₹1,00,000.
     our sources     : ['Section 1, FAQ Q007', 'Section 1, p.1']


In [148]:
import json
from pathlib import Path

p = Path("data/processed/eval_set.json")
items = json.loads(p.read_text(encoding="utf-8"))

fixes = {
    "E39": {"facts": ["5 free"]},
    "E59": {"question": "Am I liable for anything if I report third-party fraud within 3 days of the alert?",
            "facts": ["100%"]},
    "E62": {"facts": ["₹10,000", "₹1,00,000"]},
}
for it in items:
    if it["id"] in fixes:
        it.update(fixes[it["id"]])

p.write_text(json.dumps(items, ensure_ascii=False, indent=1), encoding="utf-8")
print("Corrected:", list(fixes))

Corrected: ['E39', 'E59', 'E62']


In [140]:
%pip install streamlit fastembed

  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
   ---------------------------------------- 0.0/10.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.3 MB 991.0 kB/s eta 0:00:11
   ---------------------------------------- 0.1/10.3 MB 991.0 kB/s eta 0:00:11
   ---------------------------------------- 0.1/10.3 MB 991.0 kB/s eta 0:00:11
    --------------------------------------- 0.2/10.3 MB 919.0 kB/s eta 0:00:12
    --------------------------------------- 0.2/10.3 MB 845.5 kB/s eta 0:00:12
   - -------------------------------------- 0.3/10.3 MB 1.2 MB/s eta 0:00:09
   - -------------------------------------- 0.4/10.3 MB 1.3 MB/s eta 0:00:08
   -- ------------------------------------- 0.6/10.3 MB 1.9 MB/s eta 0:00:06
   -- ------------------------------------- 0.8/10.3 MB 1.9 MB/s eta 0:00:05
   --- ------------------------------------ 1.0/10.3 MB 2.2 MB/s eta 0:00:05
   ---- ------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\PROJECTS\ARTH Assessment\finbase-rag\venv\Scripts\python.exe -m pip install --upgrade pip


In [142]:
import importlib
import rag_core
importlib.reload(rag_core)

engine = rag_core.RagEngine()
for q in ["What is the foreign currency markup on the Luxe card?", "What is the capital of France?"]:
    r = engine.answer(q)
    print("\nQ:", q)
    print("A:", r["answer"])
    print("Sources:", [c["citation"].split("— ")[-1] for c in r["sources"]])
    print(f"found={r['found']} sim={r['top_sim']:.2f} model={r['model']} {r['seconds']:.1f}s")


2026-10-09 12:21:29,574 HTTP Request: GET https://huggingface.co/api/models/Qdrant/bge-small-en-v1.5-onnx-Q "HTTP/1.1 200 OK"
2026-10-09 12:21:29,866 HTTP Request: GET https://huggingface.co/api/models/Qdrant/bge-small-en-v1.5-onnx-Q/tree/aa8f8b060edb00e03bfdd08813a2949946c8ba55?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-10-09 12:21:30,332 HTTP Request: GET https://huggingface.co/api/models/Qdrant/bge-small-en-v1.5-onnx-Q/revision/main "HTTP/1.1 200 OK"
2026-10-09 12:21:30,629 HTTP Request: GET https://huggingface.co/api/models/Qdrant/bge-small-en-v1.5-onnx-Q/tree/aa8f8b060edb00e03bfdd08813a2949946c8ba55?recursive=true&expand=false "HTTP/1.1 200 OK"


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

2026-10-09 12:21:31,146 HTTP Request: HEAD https://huggingface.co/Qdrant/bge-small-en-v1.5-onnx-Q/resolve/aa8f8b060edb00e03bfdd08813a2949946c8ba55/model_optimized.onnx "HTTP/1.1 302 Found"
C:\PROJECTS\ARTH Assessment\finbase-rag\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Aashima Bhatia\AppData\Local\Temp\fastembed_cache. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/


Q: What is the foreign currency markup on the Luxe card?
A: As stated in the Credit Cards Comprehensive Handbook & Cardholder Agreement, Section 1.2, the foreign currency markup fee on the FinBase Luxe card is 1.50% on all international transactions.
Sources: ['Section 1.2, FAQ Q001', 'Section 1.2, p.1']
found=True sim=0.86 model=gemini-flash-lite-latest 2.5s

Q: What is the capital of France?
A: I couldn't find this in the FinBase knowledge base. Please contact FinBase customer support for help with this.
Sources: []
found=False sim=0.45 model=None 0.1s


In [146]:
p = Path("rag_core.py")
s = p.read_text(encoding="utf-8")

old1 = "TOP_K = 5\n"
new1 = 'TOP_K = 5\nDEFAULT_MODE = os.getenv("RETRIEVAL_MODE", "vector")   # chosen from the evaluation: vector > hybrid > bm25\n'
old2 = 'def answer(self, question, history=None, k=TOP_K, mode="hybrid"):'
new2 = 'def answer(self, question, history=None, k=TOP_K, mode=DEFAULT_MODE):'

assert s.count(old1) == 1 and s.count(old2) == 1, "Patch targets not found, tell me what rag_core.py looks like"
p.write_text(s.replace(old1, new1).replace(old2, new2), encoding="utf-8")
print("rag_core.py updated: default retrieval mode is now vector")

rag_core.py updated: default retrieval mode is now vector


In [150]:
import importlib
import rag_core
importlib.reload(rag_core)
engine = rag_core.RagEngine()

q = "Am I liable if I report third-party fraud within 3 days?"


def show(title, res):
    print("\n" + "=" * 70)
    print(title)
    print("searched as:", res["standalone"])
    print("found:", res["found"], "| reason:", res["reason"] or "ok", "| top_sim:", round(res["top_sim"], 2))
    print("answer:", res["answer"][:250])
    for h in res["retrieved"]:
        print(f"  {h['vec_sim']:.2f} | {h['chunk']['doc_stem']} | {h['chunk']['citation'].split('— ')[-1]}")


show("Fresh chat", engine.answer(q))

for prev_q in ["What is the late payment fee on a credit card?",
               "What is the interest rate for senior citizens on a 1-year FD?",
               "My UPI payment failed but money was deducted. What now?"]:
    show(f"After: {prev_q}", engine.answer(q, [{"q": prev_q, "a": "Here are the details."}]))

print("\n\nKYC Section 5 chunks:")
for c in engine.chunks:
    if c["doc_stem"] == "sample_6" and c.get("section_id") == "5":
        print(c["citation"])
        print(c["text"][:900])
        print("-" * 60)

2026-10-09 12:51:31,583 Engine ready: 120 chunks
2026-10-09 12:51:32,121 AFC is enabled with max remote calls: 10.
2026-10-09 12:51:34,150 HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-lite-latest:generateContent "HTTP/1.1 200 OK"
2026-10-09 12:51:34,166 q='Am I liable if I report third-party fraud within 3 days?' found=True sim=0.78 reason=ok 2.2s
2026-10-09 12:51:34,170 AFC is enabled with max remote calls: 10.



Fresh chat
searched as: Am I liable if I report third-party fraud within 3 days?
found: True | reason: ok | top_sim: 0.78
answer: Under the KYC Verification, Compliance & Security Master Document, Section 5, you have Zero Liability (100% refund) if third-party electronic fraud is reported within 3 calendar days of receiving the transaction alert.
  0.78 | sample_6 | Section 5, FAQ Q004
  0.68 | sample_6 | Section 5, p.2
  0.64 | sample_2 | Section 3, FAQ Q005
  0.63 | sample_2 | Section 3, p.2
  0.63 | sample_2 | Section 2, FAQ Q002


2026-10-09 12:51:35,384 HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-lite-latest:generateContent "HTTP/1.1 200 OK"
2026-10-09 12:51:35,448 AFC is enabled with max remote calls: 10.
2026-10-09 12:51:37,032 HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-lite-latest:generateContent "HTTP/1.1 200 OK"
2026-10-09 12:51:37,141 q='Am I liable if I report third-party fraud within 3 days?' found=True sim=0.78 reason=ok 3.0s
2026-10-09 12:51:37,144 AFC is enabled with max remote calls: 10.



After: What is the late payment fee on a credit card?
searched as: Am I liable if I report third-party fraud within 3 days?
found: True | reason: ok | top_sim: 0.78
answer: Under the KYC Verification, Compliance & Security Master Document, Section 5, you have Zero Liability (100% refund) if third-party electronic fraud is reported within 3 calendar days of receiving the transaction alert.
  0.78 | sample_6 | Section 5, FAQ Q004
  0.68 | sample_6 | Section 5, p.2
  0.64 | sample_2 | Section 3, FAQ Q005
  0.63 | sample_2 | Section 3, p.2
  0.63 | sample_2 | Section 2, FAQ Q002


2026-10-09 12:51:38,394 HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-lite-latest:generateContent "HTTP/1.1 200 OK"
2026-10-09 12:51:38,437 AFC is enabled with max remote calls: 10.
2026-10-09 12:51:40,790 HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-lite-latest:generateContent "HTTP/1.1 200 OK"
2026-10-09 12:51:40,823 q='Am I liable if I report third-party fraud within 3 days?' found=True sim=0.78 reason=ok 3.7s
2026-10-09 12:51:40,829 AFC is enabled with max remote calls: 10.



After: What is the interest rate for senior citizens on a 1-year FD?
searched as: Am I liable if I report third-party fraud within 3 days?
found: True | reason: ok | top_sim: 0.78
answer: Under the KYC Verification, Compliance & Security Master Document, Section 5, you have Zero Liability (100% refund) if third-party electronic fraud is reported within 3 calendar days of receiving the transaction alert.
  0.78 | sample_6 | Section 5, FAQ Q004
  0.68 | sample_6 | Section 5, p.2
  0.64 | sample_2 | Section 3, FAQ Q005
  0.63 | sample_2 | Section 3, p.2
  0.63 | sample_2 | Section 2, FAQ Q002


2026-10-09 12:51:42,241 HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-lite-latest:generateContent "HTTP/1.1 200 OK"
2026-10-09 12:51:42,316 AFC is enabled with max remote calls: 10.
2026-10-09 12:51:43,713 HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-lite-latest:generateContent "HTTP/1.1 200 OK"
2026-10-09 12:51:43,719 q='Am I liable if I report third-party fraud within 3 days?' found=True sim=0.78 reason=ok 2.9s



After: My UPI payment failed but money was deducted. What now?
searched as: Am I liable if I report third-party fraud within 3 days?
found: True | reason: ok | top_sim: 0.78
answer: Under KYC Verification, Compliance & Security Master Document, Section 5, you have Zero Liability (100% refund) if third-party electronic fraud is reported within 3 calendar days of receiving the transaction alert.
  0.78 | sample_6 | Section 5, FAQ Q004
  0.68 | sample_6 | Section 5, p.2
  0.64 | sample_2 | Section 3, FAQ Q005
  0.63 | sample_2 | Section 3, p.2
  0.63 | sample_2 | Section 2, FAQ Q002


KYC Section 5 chunks:
KYC Verification, Compliance & Security Master Document (FB-POL-KYC-SEC-2026-V5) — Section 5, p.2
KYC Verification, Compliance & Security Master Document (FB-POL-KYC-SEC-2026-V5) | Section 5: Electronic Fraud & Zero Liability Policy
FinBase enforces RBI Customer Protection Guidelines regarding unauthorized electronic banking transactions:
• Zero Liability (100% Refund):
• Applies when 